# 📊 Notebook 03 — The Spreadsheet of the Mind: Dynamic Programming

**Series**: RL Notebook Series · Act I — Mathematical Foundations · Post 3 of 15

---

## What You'll Learn

Notebook 02 gave us the Bellman equation — a constraint that V must satisfy. But it didn't tell us *how to compute* V.

**Dynamic Programming** fills that gap. It's a family of algorithms that use the Bellman equation *iteratively* to compute value functions and optimal policies — as long as we have access to the environment model (the transition probabilities P and rewards R).

By the end, you'll understand:

1. **Policy Evaluation** — computing V for a fixed policy
2. **Policy Improvement** — making a policy greedy with respect to V
3. **Policy Iteration** — alternating between the two until convergence
4. **Value Iteration** — collapsing the two steps into one
5. **Why DP breaks down** at scale — and what comes next


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})
np.random.seed(42)

from utils.envs import GridEnvironment, ChainMDP
from utils.plotting import plot_values_grid, plot_policy_arrows

## 1. The Big Idea: Iterating Towards Truth

Recall from Notebook 02 that the Bellman Expectation Equation says:

```
V(s) = r(s) + γ · Σ P(s→s') · V(s')
```

The problem: to compute V(s) we need V(s') — a circular dependency.

Dynamic Programming's trick: **start with a wrong guess and keep correcting it.**

Initialise V(s) = 0 for all states. Then repeatedly apply the Bellman equation as an *update rule*:

```
V_new(s) ← r(s) + γ · Σ P(s→s') · V_old(s')
```

Each sweep makes the estimate a little more accurate. Eventually it converges to the true V.

**The spreadsheet analogy:** imagine a spreadsheet where every cell depends on its neighbours. If you set all cells to 0 and press F9 (recalculate), the values propagate inward sweep by sweep until every cell stabilises. That's DP.

### The Environment

We'll use the same 4×4 grid from Notebook 02:
- **S** = start (0,0), **G** = goal (3,3)
- **■** = wall at (1,1), **X** = traps at (1,3) and (3,0)
- Each step costs -0.1; reaching the goal gives +1.0; hitting a trap gives -1.0


In [ ]:
env = GridEnvironment(
    rows=4, cols=4,
    start=(0, 0), goal=(3, 3),
    walls=[(1, 1)], traps=[(1, 3), (3, 0)]
)

print("Grid layout (A=agent, G=goal, ■=wall, X=trap):")
env.render()

## 2. Policy Evaluation: Computing V for a Fixed Policy

**Goal:** given a policy π, find V^π — the expected return from each state.

**Algorithm (iterative Bellman backup):**

```
Initialise V(s) = 0 for all s
Repeat until max|V_new(s) - V(s)| < θ:
    For each s:
        a = π(s)
        V_new(s) = Σ P(s,a,s') · [r(s,a,s') + γ · V(s')]
```

The threshold θ (theta) controls convergence: we stop when the biggest change in any state's value is smaller than θ.


In [ ]:
def policy_evaluation(env, policy_fn, gamma=0.9, theta=1e-6):
    """
    Evaluate a policy by iterating the Bellman expectation equation.

    Args:
        env:       environment with env.P[s][a] = [(prob, s', reward, done), ...]
        policy_fn: callable, policy_fn(s) -> action
        gamma:     discount factor
        theta:     convergence threshold

    Returns:
        V: np.ndarray of shape (n_states,)
    """
    V = np.zeros(env.n_states)

    # ============================================================
    # TODO: Implement the iterative Bellman backup.
    #
    # Loop until the largest change in V across all states is
    # smaller than theta. For each state s:
    #   1. Look up the action from policy_fn(s)
    #   2. Compute the new value using env.P[s][a]
    #   3. Update delta and V[s]
    # ============================================================
    raise NotImplementedError

    return V

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

The Bellman backup for a fixed policy is:

    v_new = Σ prob * (reward + gamma * V[s_next])

Iterate over `env.P[s][a]` which gives `(prob, s_next, reward, done)` tuples.

Track `delta = max(delta, |v_new - V[s]|)` before updating `V[s]`.
Stop when `delta < theta`.

</details>

### Test: Evaluate a fixed 'always go right' policy

We know from Notebook 02 that on the ChainMDP, always-right gives:
V = [0.0, 0.791, 0.89, 1.0, 0.0]. Let's verify our implementation matches.


In [ ]:
chain = ChainMDP(n=5, left_reward=0.0, right_reward=1.0, step_reward=-0.01)

def always_right(s):
    return 1  # action 1 = RIGHT

V_chain = policy_evaluation(chain, always_right, gamma=0.9)
print("Policy evaluation on ChainMDP (always right):")
print(V_chain.round(4))

expected = np.array([0.0, 0.791, 0.89, 1.0, 0.0])
assert np.allclose(V_chain, expected, atol=1e-3), "Mismatch!"
print("\n✅ Matches Notebook 02 exact solution!")

<details>
<summary>✅ <b>Checkpoint — expected output</b> (click to reveal)</summary>

```
Policy evaluation on ChainMDP (always right):
[0.     0.791  0.89   1.     0.    ]
✅ Matches Notebook 02 exact solution!
```

</details>

In [ ]:
# Evaluate on the grid — smart policy from Notebook 02
smart_policy_dict = {
    0: 1, 1: 1, 2: 2, 3: 2,
    4: 2, 6: 2,
    8: 1, 9: 1, 10: 2, 11: 2,
    13: 1, 14: 1,
}

def smart_policy(s):
    return smart_policy_dict.get(s, 1)

V_smart = policy_evaluation(env, smart_policy, gamma=0.9)

fig, ax = plt.subplots(figsize=(6, 5))
plot_values_grid(V_smart, (4, 4), title="V (Smart Policy, γ=0.9)", ax=ax)
plt.tight_layout()
plt.show()
print(f"V(start=0): {V_smart[0]:.3f}")

## 3. Policy Improvement: Acting Greedily with Respect to V

Once we have an accurate V, we can improve the policy by being **greedy**:

```
π_new(s) = argmax_a [ Σ P(s,a,s') · (r(s,a,s') + γ · V(s')) ]
```

In words: for each state, pick the action whose one-step lookahead gives the highest value.

This is essentially computing Q(s, a) for all actions and picking the best one.

**The Policy Improvement Theorem** guarantees: the new policy is at least as good as the old one.


In [ ]:
def policy_improvement(env, V, gamma=0.9):
    """
    Compute a greedy policy with respect to a value function V.

    For each state, pick the action that maximises the one-step Bellman lookahead.

    Returns:
        policy: np.ndarray of shape (n_states,), dtype int
    """
    policy = np.zeros(env.n_states, dtype=int)

    # ============================================================
    # TODO: For each state s, compute the Q-value for every action
    # and store the best action in policy[s].
    #
    # Q(s, a) = Σ prob * (reward + gamma * V[s_next])
    # over all (prob, s_next, reward, done) in env.P[s][a]
    # ============================================================
    raise NotImplementedError

    return policy

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

Loop over all states `s`, then loop over all actions `a` in `range(env.n_actions)`.

For each action, compute:

    q = sum(prob * (reward + gamma * V[s_next])
            for prob, s_next, reward, done in env.P[s][a])

Then `policy[s] = argmax over all Q values`.
Use `int(np.argmax(q_values))`.

</details>

In [ ]:
improved = policy_improvement(env, V_smart, gamma=0.9)

ACTION_NAMES = {0: '↑', 1: '→', 2: '↓', 3: '←'}
print("Greedy policy (arrow for each state):")
for r in range(4):
    row = []
    for c in range(4):
        s = r * 4 + c
        if (r, c) == env.goal:
            row.append(' G ')
        elif (r, c) in env.walls:
            row.append(' ■ ')
        elif (r, c) in env.traps:
            row.append(' X ')
        else:
            row.append(f' {ACTION_NAMES[improved[s]]} ')
    print('|'.join(row))

## 4. Policy Iteration: Evaluate → Improve → Repeat

**Policy Iteration** alternates between policy evaluation and improvement:

```
Initialise π arbitrarily (e.g. all actions = 0)
Repeat:
    V ← PolicyEvaluation(π)       # find V for current policy
    π_new ← PolicyImprovement(V)  # find greedy policy
    if π_new == π: STOP           # policy is stable → optimal!
    π ← π_new
```

**Convergence guarantee:** Because there are finitely many deterministic policies and each iteration produces a strictly better (or equal) policy, Policy Iteration must converge in finite steps.


In [ ]:
def policy_iteration(env, gamma=0.9, theta=1e-6):
    """
    Find the optimal policy by alternating policy evaluation and improvement.

    Returns:
        V:       final value function (np.ndarray)
        policy:  optimal policy (np.ndarray of action indices)
        history: list of (policy, V) snapshots, one per iteration
    """
    policy = np.zeros(env.n_states, dtype=int)
    history = []

    # ============================================================
    # TODO: Implement the Policy Iteration loop.
    #
    # Each iteration:
    #   1. Evaluate the current policy (use your policy_evaluation)
    #   2. Record a snapshot in history: history.append((policy.copy(), V.copy()))
    #   3. Improve the policy (use your policy_improvement)
    #   4. Check if the policy changed. If not, break.
    # ============================================================
    raise NotImplementedError

    return V, policy, history

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

The loop is:

    while True:
        V = policy_evaluation(env, lambda s: policy[s], gamma, theta)
        history.append((policy.copy(), V.copy()))
        new_policy = policy_improvement(env, V, gamma)
        if np.all(new_policy == policy):
            break
        policy = new_policy

The convergence check `np.all(new_policy == policy)` asks: did any state's action change? If not, we've reached the optimal policy.

</details>

In [ ]:
V_pi, policy_pi, history = policy_iteration(env, gamma=0.9)

print(f"Policy Iteration converged in {len(history)} iterations.")
print(f"Optimal V(start): {V_pi[0]:.3f}")

n_iters = len(history)
fig, axes = plt.subplots(1, min(n_iters, 4), figsize=(4 * min(n_iters, 4), 4))
if n_iters == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    pol, val = history[i]
    plot_values_grid(val, (4, 4), title=f"Iter {i+1}", ax=ax)

plt.suptitle("Policy Iteration — V at each iteration", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<details>
<summary>✅ <b>Checkpoint — expected output</b> (click to reveal)</summary>

```
Policy Iteration converged in 7 iterations.
Optimal V(start): 0.181
```

</details>

## 5. Value Iteration: The Shortcut

Policy Iteration runs a full evaluation sweep to convergence between each improvement. That can be expensive.

**Value Iteration** asks: why wait for full convergence? Just do *one* evaluation update and immediately take the max over actions:

```
Initialise V(s) = 0 for all s
Repeat until max|V_new(s) - V(s)| < θ:
    For each s:
        V_new(s) = max_a [ Σ P(s,a,s') · (r(s,a,s') + γ · V(s')) ]
```

This applies the **Bellman Optimality Equation** as an update rule. Key difference from Policy Evaluation: `max_a` instead of the fixed policy's action.

Once converged, extract the greedy policy with `policy_improvement`.


In [ ]:
def value_iteration(env, gamma=0.9, theta=1e-4):
    """
    Find the optimal value function using Value Iteration.

    Returns:
        V:       optimal value function (np.ndarray)
        policy:  optimal policy derived from V* (np.ndarray of ints)
        deltas:  list of max|delta| per sweep (for convergence plot)
    """
    V = np.zeros(env.n_states)
    deltas = []

    # ============================================================
    # TODO: Implement Value Iteration.
    #
    # It's similar to policy_evaluation, but instead of using a
    # fixed action, take the MAX over all actions.
    #
    # After the loop, extract the policy using policy_improvement.
    # ============================================================
    raise NotImplementedError

    policy = policy_improvement(env, V, gamma)
    return V, policy, deltas

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

Value Iteration is almost identical to policy_evaluation, but replace:

    v_new = Σ prob * (reward + gamma * V[s_next])    # fixed action

with:

    q_values = [Σ prob*(reward+gamma*V[s_next]) for a in all actions]
    v_new = max(q_values)    # max over all actions

Track `deltas.append(delta)` each sweep so you can plot convergence.

</details>

In [ ]:
V_vi, policy_vi, deltas = value_iteration(env, gamma=0.9)

print(f"Value Iteration converged in {len(deltas)} sweeps.")
print(f"Optimal V(start): {V_vi[0]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_values_grid(V_vi, (4, 4), title="V* (Value Iteration, γ=0.9)", ax=axes[0])

axes[1].plot(deltas, color='#6366f1', linewidth=2)
axes[1].axhline(1e-4, color='#ef4444', linestyle='--', label='θ = 1e-4')
axes[1].set_xlabel('Sweep')
axes[1].set_ylabel('Max |ΔV|')
axes[1].set_title('Value Iteration Convergence')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

<details>
<summary>✅ <b>Checkpoint — expected output</b> (click to reveal)</summary>

```
Value Iteration converged in 7 sweeps.
Optimal V(start): 0.181
```

</details>

## 6. Policy Iteration vs Value Iteration

Both algorithms find the *same* optimal policy and value function — they just take different paths.

| | Policy Iteration | Value Iteration |
|---|---|---|
| Evaluation step | Full convergence (many sweeps) | One sweep |
| Improvement step | Explicit (every K sweeps) | Baked in (max over actions) |
| Iterations to convergence | Few (often < 10) | More sweeps, but each is cheap |

Let's verify they give the same answer:


In [ ]:
print("Value functions agree:", np.allclose(V_pi, V_vi, atol=1e-3))
print("Policies agree:        ", np.all(policy_pi == policy_vi))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_values_grid(V_pi, (4, 4), title="V* via Policy Iteration", ax=axes[0])
plot_values_grid(V_vi, (4, 4), title="V* via Value Iteration", ax=axes[1])
plt.tight_layout()
plt.show()

## 7. What Changes Under Stochasticity?

So far the environment has been **deterministic**. Our `GridEnvironment` supports `stochastic=True` with a `slip_prob`. Let's see how the optimal policy changes when the agent can slip.


In [ ]:
env_stoch = GridEnvironment(
    rows=4, cols=4,
    start=(0, 0), goal=(3, 3),
    walls=[(1, 1)], traps=[(1, 3), (3, 0)],
    stochastic=True, slip_prob=0.2
)

V_stoch, policy_stoch, _ = value_iteration(env_stoch, gamma=0.9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_values_grid(V_vi, (4, 4), title="V* — Deterministic", ax=axes[0])
plot_values_grid(V_stoch, (4, 4), title="V* — Stochastic (slip=20%)", ax=axes[1])
plt.tight_layout()
plt.show()

print("Deterministic V(start):", round(V_vi[0], 3))
print("Stochastic   V(start):", round(V_stoch[0], 3))
print("\nStochastic world is harder → lower values everywhere, more conservative paths")

## 8. The Curse of Dimensionality — Why DP Doesn't Scale

DP works beautifully for our 4×4 grid (16 states). But it hits a wall for real-world problems:

### The state space explosion

| Problem | States |
|---|---|
| 4×4 Grid | 16 |
| Atari frame (84×84 pixels, 256 colours) | 256^7056 ≈ 10^17000 |
| Chess positions | ~10^43 |
| Go positions | ~10^170 |

### Two further limitations

1. **Model required.** DP needs the full transition model P(s'|s,a). In most real tasks, we don't have it.
2. **Continuous spaces.** DP requires discrete, enumerable states.

### What's coming next

- **Notebook 04** solves the model-free problem: Q-Learning learns from experience without knowing P.
- **Notebook 05** solves the large-state problem: DQN replaces the value table with a neural network.


## 9. Recap

| Algorithm | What it computes | When to use |
|---|---|---|
| **Policy Evaluation** | V^π for a fixed π | Subroutine inside Policy Iteration |
| **Policy Improvement** | π greedy w.r.t. V | Subroutine; also used after Value Iteration |
| **Policy Iteration** | V*, π* | Small MDPs with cheap policy evaluation |
| **Value Iteration** | V*, π* | Preferred in practice; one-step lookahead |

### Key equations

```
Policy Evaluation:  V_new(s) = Σ_a π(a|s) · Σ_s' P(s,a,s') · [r + γ·V(s')]

Policy Improvement: π_new(s) = argmax_a [ Σ_s' P(s,a,s') · (r + γ·V(s')) ]

Value Iteration:    V_new(s) = max_a  [ Σ_s' P(s,a,s') · (r + γ·V(s')) ]
```

### Coming up

In **Notebook 04 — Learning by Stumbling**, we drop the requirement of knowing the model P. Same goal, no crystal ball.

✅ **Check your understanding:**
- Why does Policy Iteration guarantee convergence to the optimal policy?
- What's the key difference in the update rule between Policy Evaluation and Value Iteration?
- Why can't we directly apply Value Iteration to an Atari game?
